### Load environment variables

In this step, we use the dotenv package to load environment variables from a .env file. This approach helps manage sensitive configuration details like API keys and service credentials without hardcoding them in the code.

Key Points:

dotenv: Automatically loads environment variables defined in a .env file into process.env.

Access Environment Variables: The process.env object is used to access these variables in the application.

In [ ]:
import dotenv from 'dotenv';
dotenv.config();
 
console.log(process.env.AICORE_SERVICE_KEY); 

### Basic Orchestration Pipeline

In [2]:
const txtContent = await Deno.readTextFile('./support-request.txt');
console.log(txtContent);

"Subject: Bestellung #1234567890 VerspÃ¤tet - John Johnson Nachricht: Halle, ich schreibe ihnen um mich nach dem Status meiner Bestellung mit der Bestellnr. +1234567890 zu erkundigen. Die Lieferung war eigentlich fÃ¼r gestern geplant, ist bisher jedoch nicht erfolgt. Mein Name ist John Johnson und meine Lieferadresse lautet 125 Cole Meadows Drive Palo Alto, California 94301. Bitte lassen Sie mich per Telefon unter der Nummer +1 505802 2172 wissen, wann ich mit meiner Lieferung rechnen kann. Danke!"


### Prompt Templating

**LLM Configuration**

Choose the LLM by setting the name property in the promptTemplating.model configuration.

**Template Configuration**

Use the orchestration client with the promptTemplating.prompt.template configuration to define a static prompt. This prompt can include placeholders, which are replaced with values from placeholderValues during a chatCompletion() method call 

Key Components:
- **SystemMessage**: Sets a predefined instruction for the AI assistant. This message typically includes the assistant's role and any specific guidelines it should follow.
- **UserMessage**: Represents the user's input and how it is structured in the conversation.
  
In this revised prompt, only queries are passed to the assistant without any additional context. The AI is expected to respond based solely on the provided input.


In [3]:
import { OrchestrationClient } from '@sap-ai-sdk/orchestration';

const orchestrationClient = new OrchestrationClient({
  promptTemplating: {
    model: {
      name: 'gpt-4o',
      params: {
        max_completion_tokens: 200,
        temperature: 0
      }
    },
    prompt: {
      template: [
        {
          role: 'system',
          content:
            'You are a customer support assistant. Analyze the sentiment of the user request provided and return whether the sentiment is positive, neutral, or negative. Also provide a one-line justification.'
        },
        {
          role: 'user',
          content:
            'Please analyze the sentiment of the following support request: {{ ?support_text }}'
        }
      ]
    }
  }
});

const response = await orchestrationClient.chatCompletion({
  placeholderValues: {
    support_text: 'User is unhappy with the latest update and facing usability issues.'
  }
});


### Data masking

The Data Masking Module anonymizes or pseudonymizes personally identifiable information (PII) before it is processed by the LLM module. When data is anonymized, all identifying information is replaced with placeholders (e.g., MASKED_ENTITY), and the original data cannot be recovered, ensuring that no trace of the original information is retained. In contrast, pseudonymized data is substituted with unique placeholders (e.g., MASKED_ENTITY_ID), allowing the original information to be restored if needed. In both cases, the masking module identifies sensitive data and replaces it with appropriate placeholders before further processing.

In [ ]:
import { buildDpiMaskingProvider } from '@sap-ai-sdk/orchestration';

const maskingProvider = buildDpiMaskingProvider({
  method: 'anonymization',
  entities: [
    'profile-person',
    'profile-email',
    'profile-phone',
    {
      type: 'custom',
      // Example: customer / ticket reference IDs
      regex: '\\b(TICKET|CASE)-[0-9]{4,}\\b',
      replacement_strategy: {
        method: 'constant',
        value: 'MASKED_REFERENCE_ID'
      }
    }
  ],
  allowlist: ['SAP'] // Optional
});


### Content Filtering

The Content Filtering Module can be configured to filter both the input to the LLM module (input filter) and the output generated by the LLM (output filter). The module uses predefined classification services to detect inappropriate or unwanted content, allowing flexible configuration through customizable thresholds. These thresholds can be set to control the sensitivity of filtering, ensuring that content meets desired standards before it is processed or returned as output.

In [5]:
import { buildAzureContentSafetyFilter } from '@sap-ai-sdk/orchestration';

// Input filter: protects what users send (support tickets)
const inputFilter = buildAzureContentSafetyFilter('input', {
  hate: 'ALLOW_SAFE_LOW',
  self_harm: 'ALLOW_SAFE_LOW',
  sexual: 'ALLOW_SAFE_LOW',
  violence: 'ALLOW_SAFE_LOW',
  prompt_shield: true 
});

// Output filter: protects what the model returns
const outputFilter = buildAzureContentSafetyFilter('output', {
  hate: 'ALLOW_SAFE',
  self_harm: 'ALLOW_SAFE',
  sexual: 'ALLOW_SAFE',
  violence: 'ALLOW_SAFE'
});


### Translation

In [6]:
import { buildTranslationConfig } from '@sap-ai-sdk/orchestration';

const inputTranslation = buildTranslationConfig('input', {
  sourceLanguage: 'de-DE',
  targetLanguage: 'en-US'
});

const outputTranslation = buildTranslationConfig('output', {
  sourceLanguage: 'en-US',
  targetLanguage: 'de-DE'
});

// ✅ Combine them into ONE config object
const translationConfig = {
  input: inputTranslation,
  output: outputTranslation
};

console.log('✅ Translation configuration defined successfully');


✅ Translation configuration defined successfully


In [7]:
import { OrchestrationClient } from '@sap-ai-sdk/orchestration';

const orchestrationClient = new OrchestrationClient({
  resourceGroup: 'grounding',

  // Sentiment analysis prompt
  promptTemplating: {
    model: {
      name: 'gpt-4o',
      params: {
        max_completion_tokens: 200,
        temperature: 0
      }
    },
    prompt: {
      template: [
        {
          role: 'system',
          content:
            'You are a customer support assistant. Analyze the sentiment of the user request provided and return whether the sentiment is positive, neutral, or negative. Also provide a one-line justification.'
        },
        {
          role: 'user',
          content:
            'Please analyze the sentiment of the following support request: {{ ?support_text }}'
        }
      ]
    }
  },

  translation: translationConfig,

  masking: {
    masking_providers: [maskingProvider]
  },

  filtering: {
    input: {
      filters: [inputFilter]
    },
    output: {
      filters: [outputFilter]
    }
  }
});


### Generate Responses 

This step outlines the process of generating responses for a set of queries using defined llm model. The generateResponsesForModels function iterates through the llm model and executes queries to gather AI-generated responses.

Key Points:

Query Execution: Uses OrchestrationClient to generate responses for each query.

In [8]:
try {
  const response = await orchestrationClient.chatCompletion({
    placeholderValues: {
      support_text: txtContent
    }
  });

  console.log(response.getContent());

} catch (error: any) {
  console.error('❌ Error during support sentiment analysis');
  console.error(error.message);
  console.error(error.cause?.response?.data);
}


Die Stimmung der Supportanfrage ist neutral. Der Benutzer fragt einfach nach dem Status seiner Bestellung, ohne Frustration oder Unzufriedenheit auszudrücken.


In [9]:
response

OrchestrationResponse {
  rawResponse: {
    status: 200,
    statusText: "OK",
    headers: Object [AxiosHeaders] {
      date: "Thu, 05 Feb 2026 10:29:53 GMT",
      "content-type": "application/json",
      "content-length": "1955",
      "x-upstream-service-time": "693"
    },
    config: {
      transitional: {
        silentJSONParsing: true,
        forcedJSONParsing: true,
        clarifyTimeoutError: false
      },
      adapter: [ "xhr", "http", "fetch" ],
      transformRequest: [ [Function: transformRequest] ],
      transformResponse: [ [Function: transformResponse] ],
      timeout: 0,
      xsrfCookieName: "XSRF-TOKEN",
      xsrfHeaderName: "X-XSRF-TOKEN",
      maxContentLength: -1,
      maxBodyLength: -1,
      env: {
        FormData: [Function: FormData] {
          LINE_BREAK: "\r\n",
          DEFAULT_CONTENT_TYPE: "application/octet-stream"
        },
        Blob: [class Blob]
      },
      validateStatus: [Function: validateStatus],
      headers: Object [Axi